# Notebook to Compare Heuristics

In [16]:
import asyncio
import nest_asyncio

import random
import pandas as pd
from time import time

import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

In [17]:
random.seed(42)  # For accurate comparison
nest_asyncio.apply()

In [18]:
from src.heuristics import (
    welfare_greedy,
    c_fim,
    GRASP,
)

from src.diffusion_models import (
    estimate_cascade_influence,
    estimate_cascade_by_community,
)

In [19]:
from src import Loader
from pathlib import Path

path_to_networks = Path('../data/synthetic/networks')
path_to_results = Path('../../results/barbasi_albert/size_1000/')

file_name = 'barbasi_albert_1000'

In [20]:
async def main():
    loader = Loader(max_workers=4)
    graph = await loader.load(f'{path_to_networks}/{file_name}.pkl')

    communities = list(greedy_modularity_communities(graph))
    costs = nx.get_node_attributes(graph, 'node_costs')

    return graph, communities, costs


graph, communities, costs = asyncio.run(main())

## Comparison of Welfare Greedy and Budgeted Welfare Greedy (CFIM)

In [ ]:
k = 10  # number of seeds to select
alphas = [1, 0.5, 0, -1, -3, -5, -7, -9]  # inequality-aversion parameter
p = 0.05  # edge activation probability
budget = 5.0  # Total budget available
num_sims = 1000

In [ ]:
results = []

for alpha in alphas:
    start = time()
    welfare_seeds = welfare_greedy(
        graph=graph,
        communities=communities,
        k=k,
        alpha=alpha,
        probability=p,
        num_sims=num_sims,
    )
    welfare_time = time() - start

    welfare_influence = estimate_cascade_influence(
        graph=graph,
        seeds=welfare_seeds,
        probability=p,
        num_simulations=num_sims,
    )

    welfare_by_comm = estimate_cascade_by_community(
        graph=graph,
        seeds=welfare_seeds,
        probability=p,
        num_simulations=num_sims // 2,
    )
    welfare_by_comm_rounded = {k: round(v, 2) for k, v in welfare_by_comm.items()}

    start = time()
    cfim_seeds = c_fim(
        graph=graph,
        communities=communities,
        max_seeds=k,
        budget=budget,
        costs=costs,
        alpha=alpha,
        probability=p,
    )
    cfim_time = time() - start

    cfim_influence = estimate_cascade_influence(
        graph=graph,
        seeds=cfim_seeds,
        probability=p,
        num_simulations=num_sims,
    )

    cfim_by_comm = estimate_cascade_by_community(
        graph=graph,
        seeds=cfim_seeds,
        probability=p,
        num_simulations=num_sims // 2,
    )
    cfim_by_comm_rounded = {k: round(v, 2) for k, v in cfim_by_comm.items()}

    results.append(
        {
            'alpha': alpha,
            'cfim_seeds': cfim_seeds,
            'cfim_time_s': cfim_time,
            'cfim_influence_by_community': cfim_by_comm_rounded,
            'welfare_seeds': welfare_seeds,
            'welfare_time_s': welfare_time,
            'welfare_influence_by_community': welfare_by_comm_rounded,
            'org_total_influence': cfim_influence,
            'fair_total_influence': welfare_influence,
        }
    )

df = pd.DataFrame(results)
df.to_csv(f'{path_to_results}/welfare_cfim_{file_name}_{k}_results.csv', index=False)

## Comparison of Welfare Greedy and GRASP

In [21]:
k = 10  # number of seeds to select
alphas = [1, 0.5, 0, -1, -3, -5, -7, -9]  # inequality-aversion parameter
p = 0.05  # edge activation probability
budget = 5  # Total budget available
num_sims = 1000

In [22]:
results = []

for alpha in alphas:
    start = time()
    welfare_seeds = welfare_greedy(
        graph=graph,
        communities=communities,
        k=k,
        alpha=alpha,
        probability=p,
        num_sims=num_sims,
    )
    welfare_time = time() - start

    welfare_influence = estimate_cascade_influence(
        graph=graph,
        seeds=welfare_seeds,
        probability=p,
        num_simulations=num_sims,
    )

    welfare_by_comm = estimate_cascade_by_community(
        graph=graph,
        seeds=welfare_seeds,
        probability=p,
        num_simulations=num_sims // 2,
    )
    welfare_by_comm_rounded = {k: round(v, 2) for k, v in welfare_by_comm.items()}

    start = time()
    grasp_solver = GRASP(
        graph=graph,
        costs=costs,
        budget=budget,
        alpha=0,
        max_iter=50,
        num_sims=num_sims,
        max_evaluations=num_sims // 2,
        propagation_rate=p,
    )
    grasp_seeds, spread = grasp_solver.solve()

    grasp_time = time() - start

    grasp_influence = estimate_cascade_influence(
        graph=graph,
        seeds=grasp_seeds,
        probability=p,
        num_simulations=num_sims,
    )

    grasp_by_comm = estimate_cascade_by_community(
        graph=graph,
        seeds=grasp_seeds,
        probability=p,
        num_simulations=num_sims // 2,
    )
    grasp_by_comm_rounded = {k: round(v, 2) for k, v in grasp_by_comm.items()}

    results.append(
        {
            'alpha': alpha,
            'grasp_seeds': grasp_seeds,
            'grasp_time_s': grasp_time,
            'grasp_influence_by_community': grasp_by_comm_rounded,
            'welfare_seeds': welfare_seeds,
            'welfare_time_s': welfare_time,
            'welfare_influence_by_community': welfare_by_comm_rounded,
            'org_total_influence': grasp_influence,
            'fair_total_influence': welfare_influence,
        }
    )

df = pd.DataFrame(results)
df.to_csv(f'{path_to_results}/welfare_grasp_{file_name}_{k}_results.csv', index=False)


Selecting seeds: 100%|██████████| 10/10 [01:14<00:00,  7.42s/it, seeds=10, influenced={0: 0.05, 1: 0.02, 2: 0.08, 3: 0.04, 4: 0.15, 5: 0.17, 6: 0.01, 7: 0.02, 8: 0.0, 9: 0.04, 10: 0.08, 11: 0.08, 12: 0.02, 13: 0.15, 14: 0.01, 15: 0.01, 16: 0.08}]

Selecting seeds: 100%|██████████| 50/50 [00:03<00:00, 14.08it/s, seeds=7, spread=17.9, cost=4.99]

Selecting seeds: 100%|██████████| 10/10 [01:17<00:00,  7.74s/it, seeds=10, influenced={0: 0.06, 1: 0.07, 2: 0.05, 3: 0.06, 4: 0.08, 5: 0.06, 6: 0.06, 7: 0.04, 8: 0.04, 9: 0.05, 10: 0.07, 11: 0.07, 12: 0.07, 13: 0.04, 14: 0.05, 15: 0.05, 16: 0.06}]

Selecting seeds: 100%|██████████| 50/50 [00:03<00:00, 12.74it/s, seeds=7, spread=16.2, cost=4.98]

Selecting seeds: 100%|██████████| 10/10 [01:10<00:00,  7.08s/it, seeds=10, influenced={0: 0.06, 1: 0.06, 2: 0.05, 3: 0.06, 4: 0.08, 5: 0.06, 6: 0.06, 7: 0.05, 8: 0.04, 9: 0.05, 10: 0.07, 11: 0.07, 12: 0.07, 13: 0.04, 14: 0.06, 15: 0.05, 16: 0.06}]

Selecting seeds: 100%|██████████| 50/50 [00:02<00:00, 1